# Product Price Predictor
**Author: Ayush Anand**

### What this notebook does:
1. Installs all required libraries
2. Uploads your `train.csv` and `test.csv`
3. Extracts text features (16 regex features + 384-dim BERT embeddings)
4. Trains 7 ML models with 5-fold cross-validation
5. Builds a weighted ensemble
6. Downloads `test_out.csv` with final predictions

---
**Before running:** Make sure GPU is enabled.
Go to `Runtime → Change runtime type → T4 GPU → Save`

## Step 1 — Check GPU

In [ ]:
import torch
gpu_available = torch.cuda.is_available()
if gpu_available:
    print(f"GPU is ON: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")
    print("BERT encoding will be slow without GPU (~20 min instead of ~3 min)")

## Step 2 — Install Libraries

In [ ]:
!pip install -q sentence-transformers lightgbm xgboost scikit-learn pandas numpy tqdm
print("All libraries installed!")

## Step 3 — Upload Data Files
Click the upload button and select **both** files:
- `train.csv`  (from your Mac: `dataset/train.csv`)
- `test.csv`   (from your Mac: `dataset/test.csv`)

In [ ]:
from google.colab import files
print("A file picker will appear below. Select train.csv and test.csv together.")
uploaded = files.upload()
print(f"\nUploaded files: {list(uploaded.keys())}")

## Step 4 — Imports & Config

In [ ]:
import warnings, logging, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('PricePredictorTraining')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
Path('models').mkdir(exist_ok=True)

print("Imports done!")

## Step 5 — Load Data

In [ ]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
print(f"Train: {len(train_df):,} rows")
print(f"Test:  {len(test_df):,} rows")
print(f"\nSample row:")
train_df.head(2)

## Step 6 — Helper Functions (SMAPE + Text Features)

In [ ]:
import re

def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error — lower is better."""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0.01, None)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0 + 1e-8
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100)

def _extract_float(text, prefix):
    idx = text.find(prefix)
    if idx == -1: return 0.0
    snippet = text[idx+len(prefix):idx+len(prefix)+30]
    m = re.search(r'[\d]+\.?[\d]*', snippet)
    return float(m.group()) if m else 0.0

def _extract_pack(text):
    for pat in [r'pack\s+of\s+(\d+)', r'\((\d+)\s+pack\)', r'(\d+)\s*(?:count|pcs|pieces|ct\b)', r'set\s+of\s+(\d+)']:
        m = re.search(pat, text)
        if m: return float(m.group(1))
    return 1.0

def _parse_catalog(content):
    text = content.lower()
    value = _extract_float(text, 'value:')
    unit_map = {'fl oz':1.0,'oz':1.0,'ounce':1.0,'lb':16.0,'pound':16.0,'kg':35.27,
                'gram':0.035,'grams':0.035,'g ':0.035,'ml':0.034,'milliliter':0.034,
                'liter':33.8,'litre':33.8,'count':1.0,'pack':1.0,'piece':1.0,'pcs':1.0}
    unit_score = next((mult for unit, mult in unit_map.items() if unit in text), 0.0)
    pack_qty = _extract_pack(text)
    title_line = next((l for l in content.split('\n') if 'item name:' in l.lower() or l.strip()), '')
    title_words, title_chars = len(title_line.split()), len(title_line)
    total_chars, total_words = len(content), len(content.split())
    digit_ratio = sum(c.isdigit() for c in content) / max(total_chars, 1)
    brands = ['apple','samsung','sony','lg','hp','dell','nike','adidas','amazon','google','microsoft','cisco','bosch','philips']
    brand_hit = float(any(b in text for b in brands))
    cats = {'electronic':1,'cable':1,'adapter':1,'charger':1,'food':2,'sauce':2,'coffee':2,'tea':2,
            'clothing':3,'shirt':3,'dress':3,'shoes':3,'toy':4,'game':4,'supplement':5,'vitamin':5}
    category = next((v for k, v in cats.items() if k in text), 0)
    return {'value':value,'unit_score':unit_score,'pack_qty':pack_qty,'title_words':title_words,
            'title_chars':title_chars,'total_chars':total_chars,'total_words':total_words,
            'digit_ratio':digit_ratio,'brand_hit':brand_hit,'category':float(category),
            'has_size':float(any(k in text for k in ['inch','"','cm','mm','size'])),
            'has_weight':float(any(k in text for k in ['oz','lb','gram','kg'])),
            'has_volume':float(any(k in text for k in ['ml','liter','gallon','fl oz'])),
            'value_x_pack':value*max(pack_qty,1),'log_value':np.log1p(value),
            'log_total_words':np.log1p(total_words)}

def extract_text_features(df, fit_scaler=None):
    log.info('Extracting text features...')
    rows = [_parse_catalog(c) for c in df['catalog_content'].fillna('')]
    X = pd.DataFrame(rows).values.astype(np.float32)
    if fit_scaler is None:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    else:
        scaler = fit_scaler
        X = scaler.transform(X)
    log.info(f'Text features shape: {X.shape}')
    return X, scaler

print('Helper functions ready!')

## Step 7 — BERT Embeddings
This extracts the **meaning** of each product description using a pre-trained language model.
Each product gets a 384-number fingerprint of what it IS.

**Time:** ~3-5 min on GPU, ~20 min on CPU

In [ ]:
from sentence_transformers import SentenceTransformer

def extract_bert_features(df, model, batch_size=512):
    texts = df['catalog_content'].fillna('').tolist()
    log.info(f'Encoding {len(texts):,} texts with BERT...')
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    log.info(f'BERT embeddings shape: {embeddings.shape}')
    return embeddings.astype(np.float32)

print('Loading BERT model (downloads ~90MB, one time only)...')
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
print('BERT model loaded!')

In [ ]:
print('Encoding TRAIN data...')
X_train_bert = extract_bert_features(train_df, bert_model)

print('\nEncoding TEST data...')
X_test_bert = extract_bert_features(test_df, bert_model)

print(f'\nDone! Train BERT shape: {X_train_bert.shape}')
print(f'      Test  BERT shape: {X_test_bert.shape}')

## Step 8 — Text Features + Combine with BERT

In [ ]:
X_train_text, scaler = extract_text_features(train_df)
X_test_text, _       = extract_text_features(test_df, fit_scaler=scaler)

# Combine: [16 text features] + [384 BERT features] = 400 features total
X_train = np.hstack([X_train_text, X_train_bert])
X_test  = np.hstack([X_test_text,  X_test_bert])

# Target: log-transform price to handle skewed distribution
y_train = np.log1p(train_df['price'].values.astype(np.float64))

print(f'X_train: {X_train.shape}  (samples x features)')
print(f'X_test:  {X_test.shape}')
print(f'y_train: {y_train.shape}  (log-transformed prices)')
print(f'Price range: ${np.expm1(y_train.min()):.2f} – ${np.expm1(y_train.max()):.2f}')

## Step 9 — Define 7 Models

In [ ]:
MODELS = {
    # 1. Random Forest — many decision trees, each trained on random subset
    'random_forest': RandomForestRegressor(
        n_estimators=200, max_depth=20, random_state=RANDOM_SEED, n_jobs=-1),

    # 2. Extra Trees — like Random Forest but even more random (faster, sometimes better)
    'extra_trees': ExtraTreesRegressor(
        n_estimators=200, max_depth=20, random_state=RANDOM_SEED, n_jobs=-1),

    # 3. XGBoost — gradient boosting, builds trees sequentially to fix errors
    'xgboost': xgb.XGBRegressor(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0, tree_method='hist', device='cuda'),

    # 4. LightGBM — faster gradient boosting, works well on large datasets
    'lightgbm': lgb.LGBMRegressor(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        num_leaves=63, subsample=0.8,
        random_state=RANDOM_SEED, n_jobs=-1, verbose=-1, device='gpu'),

    # 5. Gradient Boosting — sklearn's own boosting implementation
    'gradient_boosting': GradientBoostingRegressor(
        n_estimators=150, max_depth=5, learning_rate=0.1,
        subsample=0.8, random_state=RANDOM_SEED),

    # 6. Ridge Regression — simple linear model with regularization
    'ridge_regression': Ridge(alpha=10.0),

    # 7. Neural Network — multi-layer perceptron (512 → 256 → 128 → 1)
    'neural_network': MLPRegressor(
        hidden_layer_sizes=(512, 256, 128), activation='relu',
        max_iter=200, early_stopping=True, validation_fraction=0.1,
        random_state=RANDOM_SEED, verbose=False),
}

print(f'{len(MODELS)} models ready to train!')
for name in MODELS:
    print(f'  - {name}')

## Step 10 — Train All Models with 5-Fold Cross Validation

**What is 5-Fold CV?**
Split training data into 5 parts. Train on 4, test on 1. Rotate 5 times.
Average SMAPE across 5 rounds = honest measure of model quality.

**Expected time:** ~25-40 min total on Colab GPU

In [ ]:
def cross_validate_model(name, model, X, y, n_folds=5):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_SEED)
    fold_smapes = []
    for fold_idx, (tr, val) in enumerate(kf.split(X)):
        X_tr, X_val = X[tr], X[val]
        y_tr, y_val = y[tr], y[val]
        model.fit(X_tr, y_tr)
        preds = np.expm1(np.clip(model.predict(X_val), 0, 15))
        s = smape(np.expm1(y_val), preds)
        fold_smapes.append(s)
        print(f'  {name} — fold {fold_idx+1}/{n_folds}: SMAPE={s:.2f}%')
    return float(np.mean(fold_smapes))


def train_all_models(X_train, y_train, X_test):
    results = {}
    final_preds = {}

    for name, model in MODELS.items():
        print(f'\n{"-"*50}')
        print(f'Training: {name.upper()}')
        print(f'{"-"*50}')
        t0 = time.time()

        try:
            cv_smape = cross_validate_model(name, model, X_train, y_train)
        except Exception as e:
            print(f'  CV failed ({e}), falling back to single split...')
            split = int(0.8 * len(X_train))
            model.fit(X_train[:split], y_train[:split])
            preds = np.expm1(np.clip(model.predict(X_train[split:]), 0, 15))
            cv_smape = smape(np.expm1(y_train[split:]), preds)

        elapsed = time.time() - t0

        # Retrain on full data
        model.fit(X_train, y_train)
        test_preds = np.expm1(np.clip(model.predict(X_test), 0, 15))

        results[name] = {'cv_smape': cv_smape, 'time_s': elapsed}
        final_preds[name] = test_preds

        # Save model
        with open(f'models/{name}_model.pkl', 'wb') as f:
            pickle.dump(model, f)

        print(f'  -> CV SMAPE: {cv_smape:.2f}%  |  Time: {elapsed:.0f}s  |  Saved.')

    # Weighted ensemble: better models get higher weight
    weights = {n: 1.0 / r['cv_smape'] for n, r in results.items()}
    total_w = sum(weights.values())
    weights = {n: w / total_w for n, w in weights.items()}

    print('\n' + '='*50)
    print('ENSEMBLE WEIGHTS (higher = better model):')
    for n, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f'  {n:25s}: {w:.3f}  (SMAPE {results[n]["cv_smape"]:.2f}%)')

    ensemble = sum(weights[n] * final_preds[n] for n in weights)
    ensemble = np.clip(ensemble, 0.01, None)
    return ensemble, results


print('Training functions ready. Run next cell to start training.')

In [ ]:
print('Starting training pipeline...')
print(f'Training on {X_train.shape[0]:,} samples x {X_train.shape[1]} features')
print('This will take ~25-40 minutes on Colab GPU\n')

ensemble_preds, model_results = train_all_models(X_train, y_train, X_test)

print('\n' + '='*50)
print('FINAL RESULTS (sorted best to worst):')
print('='*50)
for name, r in sorted(model_results.items(), key=lambda x: x[1]['cv_smape']):
    print(f'  {name:25s}  SMAPE: {r["cv_smape"]:.2f}%')

## Step 11 — Save Predictions & Download

In [ ]:
import pandas as pd

output_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': ensemble_preds
})

output_df.to_csv('test_out.csv', index=False)

print(f'Predictions saved: test_out.csv')
print(f'Total rows: {len(output_df):,}')
print(f'Price range: ${ensemble_preds.min():.2f} – ${ensemble_preds.max():.2f}')
print(f'\nSample predictions:')
print(output_df.head(10).to_string(index=False))

In [ ]:
from google.colab import files
files.download('test_out.csv')
print('Download started! Check your Downloads folder.')

## (Optional) Step 12 — Download Trained Models
Run this if you want to save your trained model files.

In [ ]:
import zipfile
with zipfile.ZipFile('trained_models.zip', 'w') as z:
    for f in Path('models').glob('*.pkl'):
        z.write(f)
files.download('trained_models.zip')
print('Models downloaded!')